In [2]:
import numpy as np
import pandas as pd
import random

In [3]:

df = pd.read_json("data.json")

'''
# Makes all possible combinations “actions” using categories and difficulties
# action = “ask a question from this category + difficulty level.”
'''
categories = df["category"].unique()
difficulties = df["expertise_lvl"].unique()
actions = []

for cat in categories:
    for diff in difficulties:
        actions.append((cat, diff))


In [4]:
# Simulate learner response

def simulate_response(action):
    category, expertise_lvl = action

    # Base performance probability by expertise_lvl
    expertise_lvl_levels = {'A1': 0.85, 'A2': 0.75, 'B1': 0.6, 'B2': 0.5, 'C1': 0.4, 'C2': 0.3} #should be adjusted
    base_prob = expertise_lvl_levels.get(expertise_lvl, 0.5)

    # Random variation
    noise = random.uniform(-0.1, 0.1) # to make user's behaviour less deterministic
    prob_correct = np.clip(base_prob + noise, 0.05, 0.95)

    # Simulate correctness
    is_correct = random.random() < prob_correct 

    # Response time (harder → slower) 2->8 seconds
    base_time = random.uniform(2, 8)  #should be adjusted
    time_scale = {'A1': 0.8, 'A2': 0.9, 'B1': 1.0, 'B2': 1.1, 'C1': 1.2, 'C2': 1.3} #should be adjusted
    response_time = base_time * time_scale.get(expertise_lvl, 1.0)

    # Attempts (more if wrong)
    '''attempts = user_actual_attempts'''
    attempts = 1 if is_correct else random.choice([2, 3]) #later comes from backend / #should be adjusted

    return is_correct, round(response_time, 2), attempts

In [5]:
# Reward function combining correctness + time + attempts

def get_reward(is_correct, response_time, attempts, expertise_lvl, max_time=10):
    # Correctness component
    reward = 1 if is_correct else -1

    # Response time component (faster = better)
    time_bonus = 1 - (response_time / max_time)
    time_bonus = np.clip(time_bonus, -1, 1)
    reward += time_bonus

    # Attempts component (penalty for retries)
    reward -= 0.5 * (attempts - 1)  # penalty of 0.5 for each extra attempt should be adjusted

    # expertise_lvl bonus (harder = more reward)
    difficulty_bonus = {'A1': 0, 'A2': 0.2, 'B1': 0.4, 'B2': 0.6, 'C1': 0.8, 'C2': 1.0} #should be adjusted
    reward += difficulty_bonus.get(expertise_lvl, 0)

    return round(reward, 3)

In [6]:
# Q-Learning setup

num_actions = len(actions)
num_states = 10 #should make dynamic

'''Q-table stores expected rewards for each (state, action)'''
Q = np.zeros((num_states, num_actions))

alpha = 0.1    # learning rate
gamma = 0.9    # discount factor
epsilon = 1.0  # exploration rate
epsilon_decay = 0.995
min_epsilon = 0.05

episodes = 5000 
accuracy = 0.5 #should make dynamic

In [7]:
# Training loop

for episode in range(episodes):
    state = int(accuracy * 10)
    state = np.clip(state, 0, 9)

    # ε-greedy action selection
    if random.random() < epsilon:
        action_index = random.randint(0, num_actions - 1)
    else:
        action_index = np.argmax(Q[state, :])
    action = actions[action_index] #Converts index into real question

    # Simulate response
    correct, response_time, attempts = simulate_response(action) #LATER COMES FROM FRONTEND

    # Compute reward
    reward = get_reward(correct, response_time, attempts, action[1])

    # Update accuracy proxy (for next state)
    accuracy = 0.9 * accuracy + 0.1 * (1 if correct else 0) # +by 0.1 should be disccused
    next_state = int(accuracy * 10)
    next_state = np.clip(next_state, 0, 9)

    # Q-learning formula
    Q[state, action_index] = Q[state, action_index] + alpha * (
        reward + gamma * np.max(Q[next_state, :]) - Q[state, action_index]
    )

    # Decay exploration
    epsilon = max(min_epsilon, epsilon * epsilon_decay)



In [8]:
# Show best learned actions

print("\n--- Learned Best Action per State ---")
for s in range(num_states):
    best_action = actions[np.argmax(Q[s, :])]
    print(f"State {s}: {best_action}")


--- Learned Best Action per State ---
State 0: ('basics', 1)
State 1: ('basics', 2)
State 2: ('hospitality', 3)
State 3: ('travel', 4)
State 4: ('household', 5)
State 5: ('food', 2)
State 6: ('household', 3)
State 7: ('basics', 2)
State 8: ('hospitality', 4)
State 9: ('basics', 1)


In [9]:
# Recommend next quiz question based on accuracy

def recommend_next_quiz(current_accuracy):
    state = int(current_accuracy * 10)
    state = np.clip(state, 0, 9)
    best_idx = np.argmax(Q[state, :])
    
    # map the best_idx from Q-table to the actual quiz action
    category, expertise_lvl = actions[best_idx]

    question = df[(df["category"] == category) & (df["expertise_lvl"] == expertise_lvl)].sample(1)


    print("\nRecommended Quiz:")
    print(question[["newari_word", "nepali_meaning", "category", "expertise_lvl", "type"]])
    return category, expertise_lvl

# try
recommend_next_quiz(0.68)


Recommended Quiz:
     newari_word nepali_meaning   category  expertise_lvl        type
1525         उगः           ओखली  household              3  vocabulary


('household', 3)